In [2]:
import pandas as pd
import numpy as np


In [3]:
# Load the dataset (replace with your actual file path/name)
df = pd.read_excel("../data/First Dataset.xlsx")

# Drop absolute duplicate rows immediately 
# (keeping the first occurrence)
initial_shape = df.shape
df = df.drop_duplicates()
print(f"Dropped {initial_shape[0] - df.shape[0]} duplicate rows.\n")

# Inspect the datatypes and missing value counts
print("--- Dataset Info ---")
print(df.info())
print("\n--- Missing Values ---")
print(df.isnull().sum())

# Preview the actual data
print("\n--- First 5 Rows ---")
print(df.head())

Dropped 1 duplicate rows.

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 60 entries, 0 to 59
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         60 non-null     int64  
 1   first_name          60 non-null     object 
 2   gender              60 non-null     object 
 3   age                 59 non-null     float64
 4   city                60 non-null     object 
 5   province            60 non-null     object 
 6   signup_date         60 non-null     object 
 7   membership_tier     60 non-null     object 
 8   purchase_count      60 non-null     int64  
 9   avg_order_value     60 non-null     float64
 10  total_spending      59 non-null     float64
 11  last_purchase_days  60 non-null     int64  
 12  payment_method      60 non-null     object 
 13  device              60 non-null     object 
 14  discount_used       60 non-null     object 
 15  returned_items  

In [4]:
# --- Structural Cleaning ---

# Identify all columns that are currently 'object' Convert them all to 'string' data type at once
text_columns = df.select_dtypes(include=['object']).columns
df[text_columns] = df[text_columns].astype("string[pyarrow]")

# Strip hidden leading/trailing spaces across all text columns at once
string_cols = df.select_dtypes(include=['string']).columns
for col in string_cols:
    df[col] = df[col].str.strip()

# Standardize text casing to fix typing mistakes 
df['first_name'] = df['first_name'].str.title()
df['city'] = df['city'].str.title()
df['province'] = df['province'].str.title()

# Make sure Gender and Tier are consistent (e.g., 'f' becomes 'F')
df['gender'] = df['gender'].str.upper()
df['membership_tier'] = df['membership_tier'].str.title()


# --- Type Conversion ---
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
df['age'] = df['age'].astype('Int64')

print("--- Updated Datatypes ---")
print(df.dtypes)

print("\n--- Missing Values Check ---")
print(df.isnull().sum())

--- Updated Datatypes ---
customer_id                     int64
first_name            string[pyarrow]
gender                string[pyarrow]
age                             Int64
city                  string[pyarrow]
province              string[pyarrow]
signup_date            datetime64[ns]
membership_tier       string[pyarrow]
purchase_count                  int64
avg_order_value               float64
total_spending                float64
last_purchase_days              int64
payment_method        string[pyarrow]
device                string[pyarrow]
discount_used         string[pyarrow]
returned_items                  int64
satisfaction_score              int64
dtype: object

--- Missing Values Check ---
customer_id           0
first_name            0
gender                0
age                   1
city                  0
province              0
signup_date           0
membership_tier       0
purchase_count        0
avg_order_value       0
total_spending        1
last_purchase_days  

In [5]:
# --- Invalid Values ---

# Catching Invalid Values (Business Logic)
# Identify rows where values violate common sense
invalid_age = (df['age'] < 0) | (df['age'] > 120)
invalid_score = (df['satisfaction_score'] < 1) | (df['satisfaction_score'] > 5)

print(f"Found {invalid_age.sum()} invalid ages.")
print(f"Found {invalid_score.sum()} invalid satisfaction scores.")

# Target the invalid rows, and set to NaN
df.loc[invalid_age, 'age'] = np.nan
df.loc[invalid_score, 'satisfaction_score'] = np.nan

Found 1 invalid ages.
Found 0 invalid satisfaction scores.


In [6]:
# ---  Handling Missing Values ---

# Fix 'age' by filling it with the median
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age)

# Verify everything is filled
print("--- Missing Values Check ---")
print(df.isnull().sum())

--- Missing Values Check ---
customer_id           0
first_name            0
gender                0
age                   0
city                  0
province              0
signup_date           0
membership_tier       0
purchase_count        0
avg_order_value       0
total_spending        1
last_purchase_days    0
payment_method        0
device                0
discount_used         0
returned_items        0
satisfaction_score    0
dtype: int64


In [7]:
# Define exactly what we expect in our categorical columns
expected_genders = ['M', 'F']
expected_tiers = ['Vip', 'Gold', 'Silver', 'Bronze']

# Create boolean masks (True if the data is INVALID)
invalid_gender = ~df['gender'].isin(expected_genders)
invalid_tier = ~df['membership_tier'].isin(expected_tiers)

# Flag Relational Errors
calculated_spending = df['purchase_count'] * df['avg_order_value']
math_mismatch = ~np.isclose(df['total_spending'], calculated_spending, atol=0.05)

# (Optional) Print out how many anomalies we are about to erase
print(f"Erasing {invalid_gender.sum()} invalid genders.")
print(f"Erasing {invalid_tier.sum()} invalid membership tiers.")
print(f"Erasing {math_mismatch.sum()} total spending mismatches.")

# Replace the anomalies with missing values directly
df.loc[invalid_gender, 'gender'] = pd.NA
df.loc[invalid_tier, 'membership_tier'] = pd.NA
df.loc[math_mismatch, 'total_spending'] = np.nan

# Fix 'total_spending' by calculating it from existing columns
# We only apply this calculation to the row(s) where total_spending is missing
missing_spending = df['total_spending'].isnull()
df.loc[missing_spending, 'total_spending'] = (
    df.loc[missing_spending, 'purchase_count'] * 
    df.loc[missing_spending, 'avg_order_value']
)


Erasing 0 invalid genders.
Erasing 0 invalid membership tiers.
Erasing 2 total spending mismatches.


In [8]:
# --- Fix Genders Based on the Most Common Occurrence ---

# Group the data by 'first_name' and find the most frequent 'gender' (the mode)
# If a name appears 3 times as 'M' and 1 time as 'F', the mode is 'M'.
# [0] ensures that if there is a 50/50 tie, it just picks the first one.
most_common_genders = df.groupby('first_name')['gender'].agg(lambda x: x.mode()[0])

# See the mapping it created (Optional, just to verify it looks right)
print("--- Automated Name-to-Gender Mapping ---")
print(most_common_genders)

--- Automated Name-to-Gender Mapping ---
first_name
Ali       F
Amir      F
Arash     F
Kimia     M
Maryam    M
Mina      M
Neda      F
Parsa     F
Reza      F
Sara      F
Sina      F
Zahra     M
Name: gender, dtype: string


In [9]:
# The validated external dictionary for name-to-gender mapping
gender_dict = {
    'Reza': 'M', 'Sina': 'M', 'Parsa': 'M', 'Kimia': 'F', 
    'Amir': 'M', 'Arash': 'M', 'Neda': 'F', 'Ali': 'M', 
    'Mina': 'F', 'Zahra': 'F', 'Maryam': 'F', 'Sara': 'F'
}

# Map the correct gender based on the first_name column
df['gender'] = df['first_name'].map(gender_dict).fillna(df['gender'])

In [10]:
# -- Handling Statistical Outliers --
# The Interquartile Range (IQR) method finds values that fall unusually 
# far above or below the middle 50% of your data.
Q1 = df['total_spending'].quantile(0.25)
Q3 = df['total_spending'].quantile(0.75)
IQR = Q3 - Q1


upper_bound = Q3 + 1.5 * IQR
outliers = df['total_spending'] > upper_bound
print(f"Found {outliers.sum()} extreme spenders (outliers).")

# Flag Relational Errors: Returns cannot exceed purchases
invalid_returns = df['returned_items'] > df['purchase_count']

# Find any values that were ALREADY missing before we started
already_missing = df['returned_items'].isna()

# THE BEST PRACTICE: Create a permanent flag column for transparency
df['returned_items_is_missing'] = invalid_returns | already_missing

# Print out how many anomalies we are about to erase
print(f"Erasing {invalid_returns.sum()} impossible return counts.")
print(f"Total flagged missing values in 'returned_items': {df['returned_items_is_missing'].sum()}")

# Replace the anomalies with missing values (np.nan)
df.loc[invalid_returns, 'returned_items'] = np.nan

Found 5 extreme spenders (outliers).
Erasing 6 impossible return counts.
Total flagged missing values in 'returned_items': 6


In [11]:
# Export the clean rows to your final dataset
df.to_excel("../data/cleaned_dataset.xlsx", index=False)
print("Exported 'cleaned_dataset.xlsx' with all valid data.")

Exported 'cleaned_dataset.xlsx' with all valid data.
